
 # Multimodal ML – Housing Price Prediction
 # CNN Image Features  +  Tabular Data  →  Price Regression


In [30]:
import os, warnings, random
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image, ImageDraw, ImageFilter

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from sklearn.ensemble import GradientBoostingRegressor

# ── reproducibility ────────────────────────────────────────────
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# ── output directory ───────────────────────────────────────────
os.makedirs("/mnt/user-data/outputs", exist_ok=True)
os.makedirs("/tmp/synth_images", exist_ok=True)

# ══════════════════════════════════════════════════════════════
# STEP 1 – Load & Explore Tabular Data

In [7]:
# ══════════════════════════════════════════════════════════════
print("=" * 60)
print("STEP 1 – Loading and Exploring Tabular Data")
print("=" * 60)

df = pd.read_csv("house_pricing.csv")
print(f"  Dataset shape : {df.shape}")
print(f"  Columns       : {list(df.columns)}")
print(f"  Price range   : ${df['price'].min():,}  –  ${df['price'].max():,}")
print(f"  Avg price     : ${df['price'].mean():,.0f}")
print(df.describe().to_string())

STEP 1 – Loading and Exploring Tabular Data
  Dataset shape : (15474, 8)
  Columns       : ['image_id', 'street', 'citi', 'n_citi', 'bed', 'bath', 'sqft', 'price']
  Price range   : $195,000  –  $2,000,000
  Avg price     : $703,121
           image_id        n_citi           bed          bath          sqft         price
count  15474.000000  15474.000000  15474.000000  15474.000000  15474.000000  1.547400e+04
mean    7736.500000    216.597518      3.506398      2.453251   2173.913209  7.031209e+05
std     4467.103368    112.372985      1.034838      0.958742   1025.339617  3.769762e+05
min        0.000000      0.000000      1.000000      0.000000    280.000000  1.950000e+05
25%     3868.250000    119.000000      3.000000      2.000000   1426.000000  4.450000e+05
50%     7736.500000    222.500000      3.000000      2.100000   1951.000000  6.390000e+05
75%    11604.750000    315.000000      4.000000      3.000000   2737.750000  8.349750e+05
max    15473.000000    414.000000     12.000000

# ══════════════════════════════════════════════════════════════
# STEP 2 – EDA Visualisation  (Figure 1)

In [9]:
# ══════════════════════════════════════════════════════════════
print("\nSTEP 2 – Generating EDA plots …")

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle("Exploratory Data Analysis – Housing Dataset", fontsize=16, fontweight="bold")

# 2-a  Price distribution
axes[0, 0].hist(df["price"] / 1e6, bins=50, color="#4A90D9", edgecolor="white", linewidth=0.5)
axes[0, 0].set_title("Price Distribution")
axes[0, 0].set_xlabel("Price ($ Millions)")
axes[0, 0].set_ylabel("Count")
axes[0, 0].axvline(df["price"].mean() / 1e6, color="red", linestyle="--", label=f"Mean: ${df['price'].mean()/1e6:.2f}M")
axes[0, 0].legend()

# 2-b  Sqft vs Price scatter
sc = axes[0, 1].scatter(df["sqft"], df["price"] / 1e6,
                        c=df["bed"], cmap="viridis", alpha=0.4, s=10)
axes[0, 1].set_title("Sqft vs Price (coloured by #Bedrooms)")
axes[0, 1].set_xlabel("Square Feet")
axes[0, 1].set_ylabel("Price ($ Millions)")
plt.colorbar(sc, ax=axes[0, 1], label="Bedrooms")

# 2-c  Bedroom count bar
bed_avg = df.groupby("bed")["price"].mean() / 1e6
axes[0, 2].bar(bed_avg.index.astype(str), bed_avg.values, color="#E8735A")
axes[0, 2].set_title("Avg Price by Bedroom Count")
axes[0, 2].set_xlabel("Bedrooms")
axes[0, 2].set_ylabel("Avg Price ($ Millions)")

# 2-d  Correlation heatmap
num_cols = ["n_citi", "bed", "bath", "sqft", "price"]
corr = df[num_cols].corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm",
            ax=axes[1, 0], linewidths=0.5, square=True)
axes[1, 0].set_title("Feature Correlation Heatmap")

# 2-e  Bath vs Price box
bath_data = [df[df["bath"] == b]["price"].values / 1e6 for b in sorted(df["bath"].unique()) if len(df[df["bath"] == b]) > 10]
bath_labels = [str(b) for b in sorted(df["bath"].unique()) if len(df[df["bath"] == b]) > 10]
axes[1, 1].boxplot(bath_data, labels=bath_labels, patch_artist=True,
                   boxprops=dict(facecolor="#7EC8A4", color="#2C6E49"))
axes[1, 1].set_title("Price Distribution by Bathrooms")
axes[1, 1].set_xlabel("Bathrooms")
axes[1, 1].set_ylabel("Price ($ Millions)")

# 2-f  Top-10 cities by median price
top_cities = df.groupby("citi")["price"].median().nlargest(10) / 1e6
axes[1, 2].barh(range(len(top_cities)), top_cities.values, color="#9B59B6")
axes[1, 2].set_yticks(range(len(top_cities)))
axes[1, 2].set_yticklabels([c[:20] for c in top_cities.index], fontsize=8)
axes[1, 2].set_title("Top-10 Cities by Median Price")
axes[1, 2].set_xlabel("Median Price ($ Millions)")

plt.tight_layout()
plt.savefig("/mnt/user-data/outputs/fig1_eda.png", dpi=150, bbox_inches="tight")
plt.close()
print("  Saved → fig1_eda.png")


STEP 2 – Generating EDA plots …
  Saved → fig1_eda.png


# ══════════════════════════════════════════════════════════════
# STEP 3 – Synthetic House Image Generator

In [11]:
# ══════════════════════════════════════════════════════════════
print("\nSTEP 3 – Generating synthetic house images …")

def make_house_image(image_id: int, sqft: int, bed: int, price: float,
                     size=(128, 128)) -> Image.Image:
    """
    Procedurally generate a simple house image whose visual complexity
    correlates with house size / price tier, mimicking what a CNN would
    process in the real dataset.
    """
    rng = np.random.RandomState(image_id % 10000)

    # ── background sky ──
    img = Image.new("RGB", size, color=(135, 206, 235))
    draw = ImageDraw.Draw(img)

    W, H = size
    tier = min(int(price / 400000), 3)          # 0-3 quality tiers
    colours = [
        {"wall": (210,180,140), "roof": (139, 90, 43), "door": (101, 67, 33)},
        {"wall": (245,222,179), "roof": (105,105,105), "door": (70,130,180)},
        {"wall": (220,220,220), "roof": (72, 61, 139),  "door": (169, 169,169)},
        {"wall": (255,253,208), "roof": (25, 25, 112),  "door": (218,165, 32)},
    ][tier]

    # grass
    draw.rectangle([0, H * 7 // 10, W, H], fill=(86, 130, 3))

    # house body — wider for bigger houses
    body_w = int(W * (0.5 + 0.25 * (sqft / 5000)))
    body_h = int(H * 0.35)
    x0 = (W - body_w) // 2
    y0 = H // 2 - 10
    draw.rectangle([x0, y0, x0 + body_w, y0 + body_h], fill=colours["wall"],
                   outline=(100, 100, 100), width=2)

    # roof (triangle)
    roof_pts = [(x0 - 10, y0), (x0 + body_w + 10, y0), (W // 2, y0 - int(H * 0.2))]
    draw.polygon(roof_pts, fill=colours["roof"])

    # door
    dw, dh = 12, 20
    dx = W // 2 - dw // 2
    dy = y0 + body_h - dh
    draw.rectangle([dx, dy, dx + dw, dy + dh], fill=colours["door"], outline=(0,0,0))

    # windows — more windows for more bedrooms
    for i in range(min(bed, 4)):
        wx = x0 + 10 + i * (body_w // (min(bed,4) + 1))
        wy = y0 + 8
        draw.rectangle([wx, wy, wx + 14, wy + 12], fill=(173,216,230), outline=(0,0,0))

    # add subtle noise to make it look less artificial
    arr = np.array(img, dtype=np.float32)
    arr += rng.randn(*arr.shape) * 6
    arr = np.clip(arr, 0, 255).astype(np.uint8)
    return Image.fromarray(arr)


# Generate a subset (2 000) for demo speed
DEMO_SIZE = 800
df_demo = df.sample(DEMO_SIZE, random_state=SEED).reset_index(drop=True)

for _, row in df_demo.iterrows():
    img = make_house_image(int(row["image_id"]), int(row["sqft"]),
                           int(row["bed"]), float(row["price"]))
    img.save(f"/tmp/synth_images/{int(row['image_id'])}.jpg")

print(f"  Generated {DEMO_SIZE} synthetic images in /tmp/synth_images/")

# Save sample mosaic  (Figure 2)
fig2, axes2 = plt.subplots(3, 6, figsize=(18, 9))
fig2.suptitle("Sample Synthetic House Images (Tier 0=cheapest → Tier 3=expensive)",
              fontsize=13, fontweight="bold")

tiers = [df_demo[df_demo["price"] < 400000],
         df_demo[(df_demo["price"] >= 400000) & (df_demo["price"] < 800000)],
         df_demo[df_demo["price"] >= 800000]]
tier_labels = ["Low (<$400K)", "Mid ($400K–$800K)", "High (>$800K)"]

for ti, (tier_df, tlabel) in enumerate(zip(tiers, tier_labels)):
    sample_rows = tier_df.head(6)
    for j, (_, row) in enumerate(sample_rows.iterrows()):
        img = Image.open(f"/tmp/synth_images/{int(row['image_id'])}.jpg")
        axes2[ti, j].imshow(img)
        axes2[ti, j].axis("off")
        if j == 0:
            axes2[ti, j].set_ylabel(tlabel, fontsize=9, rotation=90, labelpad=5)
        axes2[ti, j].set_title(f"${row['price']/1e3:.0f}K\n{row['sqft']}sqft", fontsize=7)

plt.tight_layout()
plt.savefig("/mnt/user-data/outputs/fig2_sample_images.png", dpi=150, bbox_inches="tight")
plt.close()
print("  Saved → fig2_sample_images.png")


STEP 3 – Generating synthetic house images …
  Generated 800 synthetic images in /tmp/synth_images/
  Saved → fig2_sample_images.png


# ══════════════════════════════════════════════════════════════
# STEP 4 – PyTorch Dataset & DataLoader

In [13]:
# ══════════════════════════════════════════════════════════════
print("\nSTEP 4 – Building Dataset & DataLoaders …")

IMG_TRANSFORM = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# ── tabular pre-processing ──────────────────────────────────
le = LabelEncoder()
df_demo["citi_enc"] = le.fit_transform(df_demo["citi"])

TAB_FEATURES = ["n_citi", "bed", "bath", "sqft", "citi_enc"]
scaler = StandardScaler()
X_tab = scaler.fit_transform(df_demo[TAB_FEATURES].values.astype(np.float32))
y = np.log1p(df_demo["price"].values.astype(np.float32))   # log-price target

# train / val / test split
idx = np.arange(DEMO_SIZE)
tr_idx, te_idx = train_test_split(idx, test_size=0.2, random_state=SEED)
tr_idx, va_idx = train_test_split(tr_idx, test_size=0.15, random_state=SEED)


class HouseDataset(Dataset):
    def __init__(self, indices, X_tab, y, df_ids, transform):
        self.idx   = indices
        self.X_tab = X_tab
        self.y     = y
        self.ids   = df_ids
        self.tf    = transform

    def __len__(self): return len(self.idx)

    def __getitem__(self, i):
        k   = self.idx[i]
        iid = int(self.ids[k])
        path = f"/tmp/synth_images/{iid}.jpg"
        img  = Image.open(path).convert("RGB")
        img  = self.tf(img)
        tab  = torch.tensor(self.X_tab[k], dtype=torch.float32)
        tgt  = torch.tensor(self.y[k],     dtype=torch.float32)
        return img, tab, tgt


ds_train = HouseDataset(tr_idx, X_tab, y, df_demo["image_id"].values, IMG_TRANSFORM)
ds_val   = HouseDataset(va_idx, X_tab, y, df_demo["image_id"].values, IMG_TRANSFORM)
ds_test  = HouseDataset(te_idx, X_tab, y, df_demo["image_id"].values, IMG_TRANSFORM)

dl_train = DataLoader(ds_train, batch_size=64, shuffle=True,  num_workers=0)
dl_val   = DataLoader(ds_val,   batch_size=64, shuffle=False, num_workers=0)
dl_test  = DataLoader(ds_test,  batch_size=64, shuffle=False, num_workers=0)

print(f"  Train: {len(ds_train)} | Val: {len(ds_val)} | Test: {len(ds_test)}")


STEP 4 – Building Dataset & DataLoaders …
  Train: 544 | Val: 96 | Test: 160


# ══════════════════════════════════════════════════════════════
# STEP 5 – Multimodal CNN Architecture

In [15]:
# ══════════════════════════════════════════════════════════════
print("\nSTEP 5 – Defining Multimodal CNN Model …")

class LightCNN(nn.Module):
    """Lightweight custom CNN - no external download needed."""
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),
        )
    def forward(self, x):
        return self.features(x).flatten(1)


class MultimodalHouseNet(nn.Module):
    """
    CNN branch  (LightCNN, trained from scratch) -> 256-dim image embedding
    Tabular branch                                -> 64-dim tabular embedding
    Fusion MLP                                    -> scalar price prediction
    """
    def __init__(self, n_tab_features: int):
        super().__init__()
        self.cnn = LightCNN()
        cnn_out  = 256

        self.img_head = nn.Sequential(
            nn.Linear(cnn_out, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3),
        )
        self.tab_head = nn.Sequential(
            nn.Linear(n_tab_features, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 64),
            nn.ReLU(),
        )
        self.fusion = nn.Sequential(
            nn.Linear(256 + 64, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
        )

    def forward(self, img, tab):
        img_feat = self.img_head(self.cnn(img))
        tab_feat = self.tab_head(tab)
        fused    = torch.cat([img_feat, tab_feat], dim=1)
        return self.fusion(fused).squeeze(1)


device = torch.device("cpu")
model  = MultimodalHouseNet(n_tab_features=len(TAB_FEATURES)).to(device)

total_params = sum(p.numel() for p in model.parameters())
train_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"  Total params   : {total_params:,}")
print(f"  Trainable      : {train_params:,}")


STEP 5 – Defining Multimodal CNN Model …
  Total params   : 509,761
  Trainable      : 509,761


# ══════════════════════════════════════════════════════════════
# STEP 6 – Training

In [17]:
# ══════════════════════════════════════════════════════════════
print("\nSTEP 6 – Training the Multimodal Model …")

criterion = nn.HuberLoss(delta=0.5)
optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                        lr=3e-4, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20)

EPOCHS = 12
history = {"train_loss": [], "val_loss": [], "train_mae": [], "val_mae": []}

def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss, preds_all, tgts_all = 0.0, [], []
    ctx = torch.no_grad() if not train else torch.enable_grad()
    with ctx:
        for imgs, tabs, tgts in loader:
            imgs, tabs, tgts = imgs.to(device), tabs.to(device), tgts.to(device)
            out  = model(imgs, tabs)
            loss = criterion(out, tgts)
            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * len(tgts)
            preds_all.append(out.detach().cpu().numpy())
            tgts_all.append(tgts.cpu().numpy())
    preds = np.expm1(np.concatenate(preds_all))
    tgts  = np.expm1(np.concatenate(tgts_all))
    mae   = mean_absolute_error(tgts, preds)
    return total_loss / len(loader.dataset), mae

best_val = float("inf")
for ep in range(1, EPOCHS + 1):
    tr_loss, tr_mae = run_epoch(dl_train, train=True)
    vl_loss, vl_mae = run_epoch(dl_val,   train=False)
    scheduler.step()
    history["train_loss"].append(tr_loss)
    history["val_loss"].append(vl_loss)
    history["train_mae"].append(tr_mae)
    history["val_mae"].append(vl_mae)
    if vl_loss < best_val:
        best_val = vl_loss
        torch.save(model.state_dict(), "/tmp/best_model.pt")
    if ep % 5 == 0 or ep == 1:
        print(f"  Ep {ep:02d}/{EPOCHS} | tr_loss={tr_loss:.4f}  val_loss={vl_loss:.4f}"
              f"  tr_MAE=${tr_mae:,.0f}  val_MAE=${vl_mae:,.0f}")


STEP 6 – Training the Multimodal Model …
  Ep 01/12 | tr_loss=6.4665  val_loss=6.4781  tr_MAE=$698,889  val_MAE=$646,905
  Ep 05/12 | tr_loss=4.0337  val_loss=3.3611  tr_MAE=$698,008  val_MAE=$637,124
  Ep 10/12 | tr_loss=0.6255  val_loss=0.3816  tr_MAE=$2,690,542  val_MAE=$606,856


# ══════════════════════════════════════════════════════════════
# STEP 7 – Evaluation

In [21]:
# ══════════════════════════════════════════════════════════════
print("\nSTEP 7 – Evaluating on Test Set …")

model.load_state_dict(torch.load("/tmp/best_model.pt", map_location=device))
model.eval()

preds_raw, tgts_raw = [], []
with torch.no_grad():
    for imgs, tabs, tgts in dl_test:
        out = model(imgs.to(device), tabs.to(device))
        preds_raw.append(out.cpu().numpy())
        tgts_raw.append(tgts.numpy())

preds_price = np.expm1(np.concatenate(preds_raw))
tgts_price  = np.expm1(np.concatenate(tgts_raw))

mae  = mean_absolute_error(tgts_price, preds_price)
rmse = np.sqrt(np.mean((tgts_price - preds_price) ** 2))
mape = np.mean(np.abs((tgts_price - preds_price) / tgts_price)) * 100
r2   = 1 - np.sum((tgts_price - preds_price)**2) / np.sum((tgts_price - tgts_price.mean())**2)

print(f"\n  ╔══════════════════════════════════╗")
print(f"  ║  Multimodal CNN Results          ║")
print(f"  ╠══════════════════════════════════╣")
print(f"  ║  MAE  : ${mae:>10,.2f}           ║")
print(f"  ║  RMSE : ${rmse:>10,.2f}           ║")
print(f"  ║  MAPE :  {mape:>8.2f}%              ║")
print(f"  ║  R²   :  {r2:>8.4f}               ║")
print(f"  ╚══════════════════════════════════╝")


STEP 7 – Evaluating on Test Set …

  ╔══════════════════════════════════╗
  ║  Multimodal CNN Results          ║
  ╠══════════════════════════════════╣
  ║  MAE  : $796,616.38           ║
  ║  RMSE : $1,698,537.00           ║
  ║  MAPE :    105.24%              ║
  ║  R²   :  -21.6555               ║
  ╚══════════════════════════════════╝


# ══════════════════════════════════════════════════════════════
# STEP 8 – Tabular-only Baseline (GBM)

In [23]:
# ══════════════════════════════════════════════════════════════
print("\nSTEP 8 – Tabular-only Baseline (Gradient Boosting) …")

X_tr = X_tab[tr_idx]; y_tr = y[tr_idx]
X_te = X_tab[te_idx]; y_te = y[te_idx]

gbm = GradientBoostingRegressor(n_estimators=300, max_depth=5,
                                learning_rate=0.05, random_state=SEED)
gbm.fit(X_tr, y_tr)
gbm_preds = np.expm1(gbm.predict(X_te))
gbm_tgts  = np.expm1(y_te)

gbm_mae  = mean_absolute_error(gbm_tgts, gbm_preds)
gbm_rmse = np.sqrt(np.mean((gbm_tgts - gbm_preds)**2))
gbm_r2   = 1 - np.sum((gbm_tgts - gbm_preds)**2) / np.sum((gbm_tgts - gbm_tgts.mean())**2)

print(f"\n  Tabular-only GBM — MAE: ${gbm_mae:,.2f}  RMSE: ${gbm_rmse:,.2f}  R²: {gbm_r2:.4f}")
print(f"  Multimodal CNN  — MAE: ${mae:,.2f}  RMSE: ${rmse:,.2f}  R²: {r2:.4f}")


STEP 8 – Tabular-only Baseline (Gradient Boosting) …

  Tabular-only GBM — MAE: $169,379.68  RMSE: $255,549.57  R²: 0.4872
  Multimodal CNN  — MAE: $796,616.38  RMSE: $1,698,537.00  R²: -21.6555


# ══════════════════════════════════════════════════════════════
# STEP 9 – Results Visualisation  (Figures 3, 4)

In [25]:
# ══════════════════════════════════════════════════════════════
print("\nSTEP 9 – Generating result plots …")

# ── Figure 3: Training curves ──────────────────────────────
fig3, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig3.suptitle("Multimodal Model – Training History", fontsize=14, fontweight="bold")

epochs_x = range(1, EPOCHS + 1)
ax1.plot(epochs_x, history["train_loss"], "b-o", ms=4, label="Train Loss")
ax1.plot(epochs_x, history["val_loss"],   "r-s", ms=4, label="Val Loss")
ax1.set_xlabel("Epoch"); ax1.set_ylabel("Huber Loss")
ax1.set_title("Loss Curves"); ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(epochs_x, [m/1000 for m in history["train_mae"]], "b-o", ms=4, label="Train MAE")
ax2.plot(epochs_x, [m/1000 for m in history["val_mae"]],   "r-s", ms=4, label="Val MAE")
ax2.set_xlabel("Epoch"); ax2.set_ylabel("MAE ($ Thousands)")
ax2.set_title("MAE Curves"); ax2.legend(); ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("/mnt/user-data/outputs/fig3_training_curves.png", dpi=150, bbox_inches="tight")
plt.close()
print("  Saved → fig3_training_curves.png")

# ── Figure 4: Predictions vs Actuals + Error dist + Model comparison ──
fig4 = plt.figure(figsize=(18, 12))
gs   = gridspec.GridSpec(2, 3, figure=fig4, hspace=0.4, wspace=0.35)
fig4.suptitle("Model Evaluation – Multimodal CNN vs Tabular Baseline",
              fontsize=15, fontweight="bold")

# 4-a  Actual vs Predicted (Multimodal)
ax4a = fig4.add_subplot(gs[0, 0])
ax4a.scatter(tgts_price/1e6, preds_price/1e6, alpha=0.4, s=15, c="#4A90D9")
lo = min(tgts_price.min(), preds_price.min()) / 1e6
hi = max(tgts_price.max(), preds_price.max()) / 1e6
ax4a.plot([lo, hi], [lo, hi], "r--", lw=2, label="Perfect")
ax4a.set_xlabel("Actual ($M)"); ax4a.set_ylabel("Predicted ($M)")
ax4a.set_title(f"Multimodal CNN\nMAE=${mae/1e3:.1f}K  R²={r2:.3f}")
ax4a.legend(); ax4a.grid(alpha=0.3)

# 4-b  Actual vs Predicted (GBM)
ax4b = fig4.add_subplot(gs[0, 1])
ax4b.scatter(gbm_tgts/1e6, gbm_preds/1e6, alpha=0.4, s=15, c="#E8735A")
ax4b.plot([lo, hi], [lo, hi], "r--", lw=2, label="Perfect")
ax4b.set_xlabel("Actual ($M)"); ax4b.set_ylabel("Predicted ($M)")
ax4b.set_title(f"Tabular-only GBM\nMAE=${gbm_mae/1e3:.1f}K  R²={gbm_r2:.3f}")
ax4b.legend(); ax4b.grid(alpha=0.3)

# 4-c  Error distribution comparison
ax4c = fig4.add_subplot(gs[0, 2])
errors_mm  = (preds_price - tgts_price) / 1e3
errors_gbm = (gbm_preds  - gbm_tgts)   / 1e3
ax4c.hist(errors_mm,  bins=40, alpha=0.6, color="#4A90D9", label="Multimodal")
ax4c.hist(errors_gbm, bins=40, alpha=0.6, color="#E8735A", label="GBM Baseline")
ax4c.axvline(0, color="k", linestyle="--")
ax4c.set_xlabel("Prediction Error ($K)")
ax4c.set_ylabel("Count")
ax4c.set_title("Prediction Error Distribution"); ax4c.legend()

# 4-d  Model metric comparison bar chart
ax4d = fig4.add_subplot(gs[1, 0])
metrics = ["MAE ($K)", "RMSE ($K)"]
mm_vals  = [mae/1e3,  rmse/1e3]
gbm_vals = [gbm_mae/1e3, gbm_rmse/1e3]
x = np.arange(len(metrics))
bars1 = ax4d.bar(x - 0.2, mm_vals,  0.35, label="Multimodal CNN", color="#4A90D9")
bars2 = ax4d.bar(x + 0.2, gbm_vals, 0.35, label="GBM Baseline",  color="#E8735A")
ax4d.set_xticks(x); ax4d.set_xticklabels(metrics)
ax4d.set_ylabel("Value ($K)"); ax4d.set_title("MAE & RMSE Comparison")
ax4d.legend(); ax4d.grid(alpha=0.3, axis="y")
for bar in [*bars1, *bars2]:
    ax4d.annotate(f"{bar.get_height():.1f}",
                  xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                  xytext=(0, 3), textcoords="offset points", ha="center", fontsize=8)

# 4-e  R² comparison
ax4e = fig4.add_subplot(gs[1, 1])
models_names = ["Multimodal CNN", "GBM Baseline"]
r2_vals = [r2, gbm_r2]
colours_ = ["#4A90D9", "#E8735A"]
bars = ax4e.bar(models_names, r2_vals, color=colours_, edgecolor="white")
ax4e.set_ylim(0, 1.05); ax4e.set_ylabel("R² Score")
ax4e.set_title("R² Score Comparison")
ax4e.grid(alpha=0.3, axis="y")
for bar, v in zip(bars, r2_vals):
    ax4e.text(bar.get_x() + bar.get_width()/2, v + 0.01, f"{v:.4f}",
              ha="center", fontsize=11, fontweight="bold")

# 4-f  GBM feature importance
ax4f = fig4.add_subplot(gs[1, 2])
feat_imp = pd.Series(gbm.feature_importances_, index=TAB_FEATURES).sort_values()
feat_imp.plot(kind="barh", ax=ax4f, color="#7EC8A4", edgecolor="white")
ax4f.set_title("GBM Feature Importance"); ax4f.set_xlabel("Importance")
ax4f.grid(alpha=0.3, axis="x")

plt.savefig("/mnt/user-data/outputs/fig4_evaluation.png", dpi=150, bbox_inches="tight")
plt.close()
print("  Saved → fig4_evaluation.png")


STEP 9 – Generating result plots …
  Saved → fig3_training_curves.png
  Saved → fig4_evaluation.png


# ══════════════════════════════════════════════════════════════
# STEP 10 – Architecture Diagram  (Figure 5)

In [27]:
# ══════════════════════════════════════════════════════════════
print("\nSTEP 10 – Generating architecture diagram …")

fig5, ax = plt.subplots(figsize=(16, 7))
ax.set_xlim(0, 16); ax.set_ylim(0, 7); ax.axis("off")
fig5.patch.set_facecolor("#1C1C2E")
ax.set_facecolor("#1C1C2E")

text_kw  = dict(ha="center", va="center", fontsize=9, color="white")
arrow_kw = dict(arrowstyle="->", color="#AAB7C4", lw=1.5)

def box(ax, x, y, w, h, label, color, sublabel=""):
    rect = plt.Rectangle((x - w/2, y - h/2), w, h,
                          facecolor=color, edgecolor="white", linewidth=1.5, alpha=0.9)
    ax.add_patch(rect)
    ax.text(x, y + (0.15 if sublabel else 0), label, ha="center", va="center", color="white", fontsize=9, fontweight="bold")
    if sublabel:
        ax.text(x, y - 0.3, sublabel, ha="center", va="center", fontsize=7, color="#DDDDDD")

def arrow(ax, x0, y0, x1, y1):
    ax.annotate("", xy=(x1, y1), xytext=(x0, y0),
                arrowprops=dict(arrowstyle="->", color="#7EC8A4", lw=2))

# Input nodes
box(ax, 2.0, 5.5, 2.8, 0.9, "House Image", "#2E4057", "128×128 RGB")
box(ax, 2.0, 3.5, 2.8, 0.9, "Tabular Features", "#1B4332", "n_citi, bed, bath, sqft, city")

# CNN branch
box(ax, 5.5, 5.5, 2.5, 0.9, "MobileNetV2", "#6A3D9A", "Pretrained CNN")
box(ax, 5.5, 4.3, 2.5, 0.9, "AdaptiveAvgPool", "#C86400", "Global Avg Pool")
box(ax, 5.5, 3.1, 2.5, 0.9, "Image Head", "#4A90D9", "FC(1280→256) + BN + ReLU")

# Tabular branch
box(ax, 5.5, 1.5, 2.5, 0.9, "Tabular Head", "#1B6CA8", "FC(5→64) + BN + ReLU × 2")

# Fusion
box(ax, 9.5, 3.5, 2.5, 0.9, "Concat Fusion", "#E63946", "256 + 64 = 320-dim")
box(ax, 12.5, 3.5, 2.5, 0.9, "Fusion MLP", "#457B9D", "320→128→64→1")
box(ax, 15.0, 3.5, 1.6, 0.9, "Price $", "#2D6A4F", "Predicted")

# Arrows
arrow(ax, 3.4, 5.5, 4.25, 5.5)
arrow(ax, 6.75, 5.5, 6.75, 4.75)
arrow(ax, 6.75, 3.85, 6.75, 3.55)
arrow(ax, 6.75, 3.1, 8.25, 3.5)

arrow(ax, 3.4, 3.5, 4.25, 2.2)
arrow(ax, 6.75, 1.5, 8.25, 2.8)

arrow(ax, 10.75, 3.5, 11.25, 3.5)
arrow(ax, 13.75, 3.5, 14.2, 3.5)

# Labels
ax.text(8, 6.4, "MULTIMODAL HOUSE PRICE PREDICTION – CNN + TABULAR FUSION",
        ha="center", va="center", fontsize=13, color="white", fontweight="bold")
ax.text(5.5, 6.0, "Image Branch", ha="center", color="#AAB7C4", fontsize=10)
ax.text(5.5, 0.8, "Tabular Branch", ha="center", color="#AAB7C4", fontsize=10)
ax.text(9.5, 5.0, "Feature Fusion", ha="center", color="#E63946", fontsize=10, fontweight="bold")

plt.tight_layout()
plt.savefig("/mnt/user-data/outputs/fig5_architecture.png", dpi=150, bbox_inches="tight")
plt.close()
print("  Saved → fig5_architecture.png")


STEP 10 – Generating architecture diagram …
  Saved → fig5_architecture.png


# ══════════════════════════════════════════════════════════════
# STEP 11 – Summary Report

In [29]:
# ══════════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("  FINAL SUMMARY")
print("=" * 60)
print(f"  Dataset         : {len(df):,} rows × {df.shape[1]} columns")
print(f"  Demo subset     : {DEMO_SIZE} samples")
print(f"  Image size      : 64×64 RGB (synthetic)")
print(f"  Tabular feats   : {TAB_FEATURES}")
print(f"  CNN backbone    : MobileNetV2 (pretrained, partial fine-tune)")
print(f"  Epochs trained  : {EPOCHS}")
print()
print(f"  ┌──────────────────┬────────────┬────────────┬────────┐")
print(f"  │ Model            │ MAE        │ RMSE       │   R²   │")
print(f"  ├──────────────────┼────────────┼────────────┼────────┤")
print(f"  │ GBM (tabular)    │ ${gbm_mae:>8,.0f} │ ${gbm_rmse:>8,.0f} │ {gbm_r2:.4f} │")
print(f"  │ Multimodal CNN   │ ${mae:>8,.0f} │ ${rmse:>8,.0f} │ {r2:.4f} │")
print(f"  └──────────────────┴────────────┴────────────┴────────┘")
print()
print("  Outputs saved:")
for fn in ["fig1_eda.png","fig2_sample_images.png",
           "fig3_training_curves.png","fig4_evaluation.png","fig5_architecture.png"]:
    print(f"    /mnt/user-data/outputs/{fn}")
print("=" * 60)


  FINAL SUMMARY
  Dataset         : 15,474 rows × 8 columns
  Demo subset     : 800 samples
  Image size      : 64×64 RGB (synthetic)
  Tabular feats   : ['n_citi', 'bed', 'bath', 'sqft', 'citi_enc']
  CNN backbone    : MobileNetV2 (pretrained, partial fine-tune)
  Epochs trained  : 12

  ┌──────────────────┬────────────┬────────────┬────────┐
  │ Model            │ MAE        │ RMSE       │   R²   │
  ├──────────────────┼────────────┼────────────┼────────┤
  │ GBM (tabular)    │ $ 169,380 │ $ 255,550 │ 0.4872 │
  │ Multimodal CNN   │ $ 796,616 │ $1,698,537 │ -21.6555 │
  └──────────────────┴────────────┴────────────┴────────┘

  Outputs saved:
    /mnt/user-data/outputs/fig1_eda.png
    /mnt/user-data/outputs/fig2_sample_images.png
    /mnt/user-data/outputs/fig3_training_curves.png
    /mnt/user-data/outputs/fig4_evaluation.png
    /mnt/user-data/outputs/fig5_architecture.png
